# Loan Data Pipeline - Project Driver Example

This notebook tests the general pipeline using the cleaned Project Driver loan-period panel as the input. It intentionally starts after the deal-specific raw tape cleaning, using the cleaned `df_rv` shape from `Project Driver Recreation Vehicles/Data_processing_project_driver.ipynb`.

## Input Assumption

Use either:
- an in-memory `df_rv` variable from the Project Driver notebook after the cleaning cells have run, or
- a saved cleaned panel file, such as `Project Driver Recreation Vehicles/project_driver_cleaned_panel.parquet`.

The raw origination/performance parquet files are not required here.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.6f}".format)

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from loanPipelineHelpers import *

try:
    from parentHelpers import plot_finance_style
except ImportError as exc:
    PARENT_HELPERS_IMPORT_ERROR = exc
    plot_finance_style = None

try:
    from regression import run_model_pipeline
except ImportError as exc:
    REGRESSION_IMPORT_ERROR = exc
    run_model_pipeline = None

try:
    from itables import init_notebook_mode, options, show
    init_notebook_mode(all_interactive=True)
    options.pageLength = 25
    options.lengthMenu = [10, 25, 50, 75, 100]
    options.scrollX = True
except ImportError:
    show = display

## Load Cleaned Project Driver Panel

In [ ]:
PROJECT_DRIVER_CLEANED_PANEL_PATH = PROJECT_ROOT / "Project Driver Recreation Vehicles" / "project_driver_cleaned_panel.parquet"

if "df_rv" in globals():
    project_driver_cleaned_raw = df_rv.copy()
    print("Using in-memory df_rv as cleaned Project Driver input.")
elif PROJECT_DRIVER_CLEANED_PANEL_PATH.exists():
    project_driver_cleaned_raw = read_tabular(PROJECT_DRIVER_CLEANED_PANEL_PATH)
    print(f"Loaded cleaned Project Driver panel from {PROJECT_DRIVER_CLEANED_PANEL_PATH}")
else:
    project_driver_cleaned_raw = pd.DataFrame()
    print("No cleaned Project Driver panel found yet.")
    print("Run the Project Driver cleaning notebook through the df_rv cleaning step, then either rerun this notebook in the same kernel or save df_rv to PROJECT_DRIVER_CLEANED_PANEL_PATH.")

print(f"project_driver_cleaned_raw shape: {project_driver_cleaned_raw.shape}")
show(project_driver_cleaned_raw.head(50))

## Standardize Project Driver Column Names

In [ ]:
PROJECT_DRIVER_PANEL_COLUMNS = {
    "loan_id": "LoanID",
    "origination_date": "FundingDate",
    "performance_date": "AsOfDate",
    "original_balance": "AmountFinanced",
    "current_balance": "EOM Bal",
    "bom_balance": "BOM Bal",
    "fico": "CreditScore",
    "product_type": "ProductType",
    "asset_type": "asset_type",
    "new_used": "New/Used",
    "interest_rate": "ContractRate",
    "term": "Term",
    "ltv": "Origination LTV ratio",
    "dti": "DTI Ratio",
    "vintage": "Vintage",
    "mob": "MOB",
    "days_delinquent": "DaysDelinquent",
    "chargeoff_status": "ChargeOff",
    "chargeoff_amount": "Default",
    "gross_chargeoff_balance": "Gross Charge Off Balance",
    "recovery_amount": "TotalRecoveriesBV",
    "cum_recovery": "Cum Recovery",
    "dq30_balance": "DQ30+ Bal",
    "scheduled_principal": "Sched Prin",
    "prepayment_amount": "Calc Prepay Amt",
}

PROJECT_DRIVER_NUMERIC_COLS = [
    "original_balance",
    "current_balance",
    "bom_balance",
    "fico",
    "interest_rate",
    "term",
    "ltv",
    "dti",
    "mob",
    "days_delinquent",
    "chargeoff_amount",
    "gross_chargeoff_balance",
    "recovery_amount",
    "cum_recovery",
    "dq30_balance",
    "scheduled_principal",
    "prepayment_amount",
]

PROJECT_DRIVER_REQUIRED_COLS = [
    "loan_id",
    "origination_date",
    "performance_date",
    "original_balance",
    "current_balance",
    "mob",
]

In [ ]:
project_driver_panel = standardize_columns(project_driver_cleaned_raw, PROJECT_DRIVER_PANEL_COLUMNS)
project_driver_panel = coerce_datetime(project_driver_panel, ["origination_date", "performance_date"])
project_driver_panel = coerce_numeric(project_driver_panel, PROJECT_DRIVER_NUMERIC_COLS)

if "current_balance" not in project_driver_panel.columns and "CurrentBalance" in project_driver_panel.columns:
    project_driver_panel["current_balance"] = pd.to_numeric(project_driver_panel["CurrentBalance"], errors="coerce")

if not project_driver_panel.empty and ("vintage" not in project_driver_panel.columns or project_driver_panel["vintage"].isna().all()):
    project_driver_panel = add_vintage_and_mob(
        project_driver_panel,
        origination_date_col="origination_date",
        performance_date_col="performance_date",
        vintage_freq="Q",
    )

if "fico" in project_driver_panel.columns and "fico_bucket" not in project_driver_panel.columns:
    project_driver_panel = add_fico_bucket(project_driver_panel, fico_col="fico")

missing_project_driver_cols = missing_columns(project_driver_panel, PROJECT_DRIVER_REQUIRED_COLS)

print("Missing required Project Driver columns:", missing_project_driver_cols)
print(f"project_driver_panel shape: {project_driver_panel.shape}")
show(project_driver_panel.head(50))

## Split Cleaned Panel Into Pipeline Views

`project_driver_origination` is one row per loan. `project_driver_performance` remains one row per loan-period. `project_driver_panel` is the merged panel used for cohort curves and target creation.

In [ ]:
ORIGINATION_KEEP_COLS = [
    "loan_id",
    "origination_date",
    "vintage",
    "original_balance",
    "fico",
    "fico_bucket",
    "product_type",
    "asset_type",
    "new_used",
    "interest_rate",
    "term",
    "ltv",
    "dti",
]

PERFORMANCE_KEEP_COLS = [
    "loan_id",
    "performance_date",
    "vintage",
    "mob",
    "current_balance",
    "bom_balance",
    "days_delinquent",
    "chargeoff_status",
    "chargeoff_amount",
    "gross_chargeoff_balance",
    "recovery_amount",
    "cum_recovery",
    "dq30_balance",
    "scheduled_principal",
    "prepayment_amount",
]

project_driver_origination = origination_from_panel(
    project_driver_panel,
    loan_id_col="loan_id",
    sort_cols=["mob", "performance_date"],
    keep_cols=ORIGINATION_KEEP_COLS,
)

project_driver_performance = performance_from_panel(
    project_driver_panel,
    keep_cols=PERFORMANCE_KEEP_COLS,
)

print(f"project_driver_origination shape: {project_driver_origination.shape}")
print(f"project_driver_performance shape: {project_driver_performance.shape}")
show(project_driver_origination.head(50))
show(project_driver_performance.head(50))

## Interactive Review Tables

In [ ]:
project_driver_panel_quality = data_quality_report(
    project_driver_panel,
    required_cols=PROJECT_DRIVER_REQUIRED_COLS,
    weight_col="original_balance",
)

project_driver_panel_coverage = panel_coverage_report(project_driver_panel)

project_driver_origination_summary = weighted_summary(
    project_driver_origination,
    weight_col="original_balance",
    numeric_cols=["fico", "interest_rate", "term", "ltv", "dti"],
)

project_driver_by_vintage = weighted_summary(
    project_driver_origination,
    groupby_cols=["vintage"] if "vintage" in project_driver_origination.columns else None,
    weight_col="original_balance",
    numeric_cols=["fico", "interest_rate", "term", "ltv", "dti"],
)

project_driver_fico_distribution = weighted_distribution(
    project_driver_origination,
    "fico_bucket",
    weight_col="original_balance",
    by_col="vintage" if "vintage" in project_driver_origination.columns else None,
) if "fico_bucket" in project_driver_origination.columns else pd.DataFrame()

project_driver_product_distribution = weighted_distribution(
    project_driver_origination,
    "product_type",
    weight_col="original_balance",
    by_col="vintage" if "vintage" in project_driver_origination.columns else None,
) if "product_type" in project_driver_origination.columns else pd.DataFrame()

project_driver_asset_distribution = weighted_distribution(
    project_driver_origination,
    "asset_type",
    weight_col="original_balance",
    by_col="vintage" if "vintage" in project_driver_origination.columns else None,
) if "asset_type" in project_driver_origination.columns else pd.DataFrame()

show(project_driver_panel_coverage)
show(project_driver_panel_quality)
show(project_driver_origination_summary)
show(project_driver_by_vintage)
show(project_driver_fico_distribution)
show(project_driver_product_distribution)
show(project_driver_asset_distribution)

## Performance Rollup

In [ ]:
project_driver_rollup = performance_rollup(
    project_driver_panel,
    chargeoff_amount_col="chargeoff_amount",
    recovery_amount_col="recovery_amount",
)

if not project_driver_rollup.empty and "chargeoff_amount" in project_driver_rollup.columns:
    project_driver_rollup["chargeoff_pct"] = project_driver_rollup["chargeoff_amount"] / project_driver_rollup["original_balance"]

show(project_driver_rollup.head(150))

## Data Wrangler Staging

Open these DataFrames in Data Wrangler for interactive inspection and transformation code generation.

In [ ]:
data_wrangler_project_driver_panel = project_driver_panel.copy()
data_wrangler_project_driver_origination = project_driver_origination.copy()
data_wrangler_project_driver_performance = project_driver_performance.copy()

for name, df in {
    "data_wrangler_project_driver_panel": data_wrangler_project_driver_panel,
    "data_wrangler_project_driver_origination": data_wrangler_project_driver_origination,
    "data_wrangler_project_driver_performance": data_wrangler_project_driver_performance,
}.items():
    print(f"{name}: {df.shape}")

## Regression-Ready Loan-Level Table

In [ ]:
DQ_THRESHOLD = 60
TARGET_CUTOFF_MOB = 12
MODEL_TARGET_COL = f"ever_dq{DQ_THRESHOLD}_or_co_mob{TARGET_CUTOFF_MOB}"

if not project_driver_panel.empty:
    project_driver_target = ever_dq_target_from_panel(
        project_driver_panel,
        dq_threshold=DQ_THRESHOLD,
        cutoff_mob=TARGET_CUTOFF_MOB,
        chargeoff_amount_col="chargeoff_amount",
        target_col=MODEL_TARGET_COL,
    )
    project_driver_regression_df = project_driver_origination.merge(
        project_driver_target[["loan_id", MODEL_TARGET_COL]],
        on="loan_id",
        how="inner",
    )
else:
    project_driver_target = pd.DataFrame()
    project_driver_regression_df = pd.DataFrame()

MODEL_START_DATE = pd.Timestamp("2022-01-01")
MODEL_END_DATE = pd.Timestamp("2025-01-31")

if not project_driver_regression_df.empty and "origination_date" in project_driver_regression_df.columns:
    project_driver_regression_train = project_driver_regression_df[
        (project_driver_regression_df["origination_date"] > MODEL_START_DATE)
        & (project_driver_regression_df["origination_date"] <= MODEL_END_DATE)
    ].copy()
else:
    project_driver_regression_train = project_driver_regression_df.copy()

print(f"project_driver_target shape: {project_driver_target.shape}")
print(f"project_driver_regression_df shape: {project_driver_regression_df.shape}")
print(f"project_driver_regression_train shape: {project_driver_regression_train.shape}")
show(project_driver_regression_train.head(100))

## Optional Regression Run

Leave `RUN_REGRESSION = False` while reviewing the data. Turn it on after the target and feature list are confirmed.

In [ ]:
RUN_REGRESSION = False

CATEGORICAL_FEATURES = ["product_type", "new_used"]
NUMERIC_FEATURES = ["fico", "interest_rate", "term", "ltv", "dti", "original_balance"]

available_categorical_features = [col for col in CATEGORICAL_FEATURES if col in project_driver_regression_train.columns]
available_numeric_features = [col for col in NUMERIC_FEATURES if col in project_driver_regression_train.columns]

if RUN_REGRESSION and run_model_pipeline is None:
    project_driver_regression_results = None
    print(f"regression.py could not be imported: {REGRESSION_IMPORT_ERROR}")
elif RUN_REGRESSION and not project_driver_regression_train.empty:
    project_driver_regression_results = run_model_pipeline(
        df=project_driver_regression_train,
        target_col=MODEL_TARGET_COL,
        categorical_features=available_categorical_features,
        numeric_features=available_numeric_features,
        model_type="logistic",
        test_size=0.2,
        id_col="loan_id",
        plot=True,
    )
else:
    project_driver_regression_results = None
    print("Regression is staged but not run. Set RUN_REGRESSION = True after reviewing the data.")

print("Categorical features:", available_categorical_features)
print("Numeric features:", available_numeric_features)

## Post-Regression Review

After reviewing `project_driver_coef_df`, add only the useful drivers to `POST_REGRESSION_CHARTS`. Each selected driver gets an actual-vs-model chart by vintage.

In [ ]:
project_driver_coef_df = (
    project_driver_regression_results["coef_df"].copy()
    if project_driver_regression_results is not None and "coef_df" in project_driver_regression_results
    else pd.DataFrame()
)

show(project_driver_coef_df)

POST_REGRESSION_GROUP_COL = "vintage"
POST_REGRESSION_WEIGHT_COL = "original_balance"

POST_REGRESSION_CHARTS = [
    # Add/remove these after reviewing project_driver_coef_df, for example:
    # {"label": "FICO", "column": "fico", "kind": "weighted_average"},
    # {"label": "LTV", "column": "ltv", "kind": "weighted_average"},
    # {"label": "Original Balance", "column": "original_balance", "kind": "weighted_average"},
    # {"label": "% New", "column": "new_used", "kind": "category_mix", "category": "New"},
    # {"label": "% Marine", "column": "product_type", "kind": "category_mix", "category": "Marine"},
]

if project_driver_regression_results is not None and POST_REGRESSION_GROUP_COL in project_driver_regression_train.columns:
    project_driver_post_regression_review_table = build_post_regression_review_table(
        project_driver_regression_results,
        reference_df=project_driver_regression_train,
        group_col=POST_REGRESSION_GROUP_COL,
        weight_col=POST_REGRESSION_WEIGHT_COL,
        chart_configs=POST_REGRESSION_CHARTS,
        id_col="loan_id",
        factor_reference_df=project_driver_origination,
    )
else:
    project_driver_post_regression_review_table = pd.DataFrame()

show(project_driver_post_regression_review_table)

if not project_driver_post_regression_review_table.empty and POST_REGRESSION_CHARTS and plot_finance_style is not None:
    project_driver_post_regression_chart_tables = plot_post_regression_review_charts(
        project_driver_post_regression_review_table,
        chart_configs=POST_REGRESSION_CHARTS,
        plot_func=plot_finance_style,
        outcome_label=f"Ever D{DQ_THRESHOLD} / CO @ MOB {TARGET_CUTOFF_MOB}",
    )
else:
    project_driver_post_regression_chart_tables = {}
    print("No post-regression charts selected yet, or plotting is unavailable.")